## Initialization

### Imports/Typing

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal, TypeVar
from random import sample
from statistics import mean
from math import ceil
import time

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
import pandas as pd
import bottleneck
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve
from data_processing.processing.slice_fitting.slice_fitters import SliceFitter
from data_processing.types import NumberedSlice, SliceFitResult, FitResult, FitErrorResult, BimodalParams
from data_processing.processing.figure_of_merit import FOM
from data_processing.processing.slice_fitting.helpers import split_params, unpack_slice_fit_pool_results
from data_processing.processing.slice_fitting.bimodal_fitting import get_bimodal_fit, get_bimodal_fit_guess
from data_processing.dataframe_validation import FIT_COLUMN_NAMES, FIT_ERROR_COLUMN_NAMES
from data_processing.helpers import get_midpoints_from_bins
# from multiprocessing.pool import Pool
from joblib import Parallel, delayed

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_ahead(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((mov_avg, prefix))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))


def moving_max(arr, n=5):
    pass  # STUB


def moving_min(arr, n=5):
    pass  # STUB

In [ ]:
def flip_pulses(pulses: pd.DataFrame) -> pd.DataFrame:
    if pulses.shape[1] != 200:
        raise ValueError("Pulse must be 200 values long")

    flipped_pulses = pulses.astype("uint32")
    pulses_np = flipped_pulses.to_numpy()
    baselines = pulses_np[:, :30].mean(axis=1).reshape(-1, 1)
    pulses_np = -pulses_np + baselines

    flipped_pulses = pd.DataFrame(
        pulses_np,
        index=flipped_pulses.index,
        columns=flipped_pulses.columns.map(int)
    )
    return flipped_pulses


def integrate_pulses(pulses: pd.DataFrame, t1: int, t2: int, t3: int) -> pd.DataFrame:
    baseline_adjust = 0
    baseline_window = 25
    
    if not pd.api.types.is_numeric_dtype(pulses.values):
        raise ValueError("DataFrame values must all be numeric type")
    if pulses.shape[1] != 200:
        raise ValueError("DataFrame rows must be 200 samples long")
    if not all([0 <= x <= 398 for x in [t1, t2, t3]]):
        raise ValueError("Times must all be between 0 and 398 (inclusive)")
    if not (t2 > t1):
        raise ValueError("t2 must be greater than t1")
    if not (t3 > t2):
        raise ValueError("t3 must be greater than t2")

    idx_1 = ceil(t1 / 2)
    idx_2 = ceil(t2 / 2)
    idx_3 = ceil(t3 / 2)

    pulses_np = pulses.to_numpy()
    baselines = np.trunc(pulses_np[:, :25].mean(axis=1).reshape(-1, 1))
    baselines = baselines + baseline_adjust
    long_slices = pulses_np[:, idx_1:idx_3]
    short_slices = pulses_np[:, idx_1:idx_2]
    peak_slices = pulses_np[:, 30:60]

    q_long = (baselines - long_slices).sum(axis=1)
    q_short = (baselines - short_slices).sum(axis=1)
    peak_heights = (baselines - peak_slices).max(axis=1)

    psd_df = pd.DataFrame(
        {"Q_LONG": q_long, "Q_SHORT": q_short, "PEAK_HEIGHT": peak_heights},
        index=pulses.index
    )
    return psd_df

In [ ]:
def calculate_psd(psd_df: pd.DataFrame) -> pd.DataFrame:
    if "Q_LONG" not in psd_df.columns:
        raise ValueError("DataFrame must have Q_LONG column")
    if "Q_SHORT" not in psd_df.columns:
        raise ValueError("DataFrame must have Q_SHORT column")
    
    q_long = psd_df["Q_LONG"]
    q_short = psd_df["Q_SHORT"]
    if not pd.api.types.is_numeric_dtype(q_long):
        raise ValueError("Q_LONG column must have numeric type")
    if not pd.api.types.is_numeric_dtype(q_short):
        raise ValueError("Q_SHORT column must have numeric type")
    
    psd_tail = q_long - q_short
    psd_df["PSD"] = psd_tail / q_long
    return psd_df

In [ ]:
def get_psd_energy_histogram(
    df: pd.DataFrame,
    energy_width: float = 400,
    energy_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = df["Q_LONG"]
    y = df["PSD"]

    within_psd = y.between(psd_min, psd_max)
    y = y[within_psd].copy()
    x = x[within_psd].copy()

    if energy_bins is not None:
        x_bins = energy_bins
    else:
        x_bins: np.ndarray = np.linspace(0, x.max(), int(x.max() / energy_width) + 1)
    # print(f"Energy width = {x_bins[1]-x_bins[0]} MeVee")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
class PeakFinderSliceFitter(SliceFitter):
    def __call__(self, numbered_slice: NumberedSlice) -> SliceFitResult:
        i, slice = numbered_slice
        slice_left_edge = self.energy_bin_edges[i]
        slice_right_edge = self.energy_bin_edges[i + 1]

        guess = get_bimodal_fit_guess(self.psd_bin_midpoints, slice)
        try:
            gamma_params, neutron_params, cov = get_bimodal_fit(
                self.psd_bin_midpoints, slice, guess=guess
            )
        except RuntimeError:
            fit_result = FitResult(
                i, None, None, slice_left_edge, slice_right_edge, None
            )
            fit_error_result = FitErrorResult(
                i, None, None, slice_left_edge, slice_right_edge
            )
            return fit_result, fit_error_result

        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])
        perr = BimodalParams(*np.sqrt(np.diag(cov)))
        fit_result = FitResult(
            i, gamma_params, neutron_params, slice_left_edge, slice_right_edge, fom
        )
        fit_error_result = FitErrorResult(
            i, *split_params(perr), slice_left_edge, slice_right_edge
        )

        return fit_result, fit_error_result


class SliceFitterFactory:
    def make_slice_fitter(
        self,
        psd_bin_midpoints: np.ndarray,
        energy_bin_edges: np.ndarray,
    ) -> SliceFitter:
        return PeakFinderSliceFitter(
            psd_bin_midpoints, energy_bin_edges, None, None
        )


def scan_histogram_slices(
    histogram: np.ndarray,
    energy_bin_edges: np.ndarray,
    psd_bin_edges: np.ndarray,
    start_idx: int = 0,
    end_idx: int | None = None,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Determines bimodal fit and FOM for every energy slice
    in a 2D PSD/Energy histogram

    Parameters
    ----------
    histogram: ndarray
        2D PSD/Energy histogram.
        The histogram shape should be [N, M], where N is the number of energy bins, and M is the number of PSD bins.
        This matches the output of Numpy's histogram2d function.
    energy_bin_edges: ndarray
        Edge values for each energy bin in the histogram.
        For N energy bins, there must be N+1 edges.
    psd_bin_edges: ndarray
        Edge values for each PSD bin in the histogram.
        For M PSD bins, there must be M+1 edges.
    fit_style: SliceFitStyle
        The style of best fit to use
    default_bounds: BimodalBounds
        Default lower and upper bounds of fit parameters
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None, default None
        Allows custom bounds for slice ranges.
        Each list entry must have a tuple of start and stop indexes, and corresponding fit bounds.
        Bounds are used when the slice index falls within the start/stop range (start inclusive, stop exclusive).
        If index ranges overlap, the last matching range is used.
        If bounds is None, only default_bounds are used.
    start_idx: int, default 0
        Starting index (inclusive) of slice range to fit to bimodal
    end_idx: int | None, default None
        Ending index (exclusive) of slice range to fit to bimodal
    cores: int, default 4
        Number of logical cores present on this computer.
        Used to control parallelization of the scan.
    use_chunks: bool, default False
        Whether to split slices into larger chunks during parallelization.
        This can help speed up the scan on larger histograms.

    Returns
    -------
    fit_dataframe: DataFrame
        DataFrame of fit parameters including FOM (as columns) for each slice (as rows)
    error_dataframe: DataFrame
        DataFrame of (1 standard deviation) errors in fit parameters (as columns) for each slice (as rows)
    """
    end_idx = len(histogram) if end_idx is None else min(len(histogram), end_idx)
    pool_size = max(
        2 * cores, 4
    )  # based on https://jupyter-tutorial.readthedocs.io/en/stable/performance/multiprocessing.html

    # psd_bin_left_edges = psd_bin_edges[:-1]
    # psd_bin_right_edges = psd_bin_edges[1:]
    # psd_bin_centers = (psd_bin_right_edges + psd_bin_left_edges) / 2
    psd_bin_centers = get_midpoints_from_bins(psd_bin_edges)
    energy_bin_edges_limited = energy_bin_edges[start_idx : end_idx + 1]

    energy_slices = list(histogram[start_idx:end_idx, :])

    if use_chunks:
        chunksize, extra = divmod(len(energy_slices), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = None

    # pool = Pool(pool_size)
    slice_fitter_factory = SliceFitterFactory()
    slice_fitter = slice_fitter_factory.make_slice_fitter(
        psd_bin_centers, energy_bin_edges_limited
    )
    # slice_fitter_factory = TestSliceFitterFactory()
    # slice_fitter = slice_fitter_factory.make_slice_fitter(0)
    # results = pool.imap_unordered(
    #     slice_fitter,
    #     enumerate(energy_slices),
    #     chunksize=chunksize,
    # )
    # results = [slice_fitter(num_slice) for num_slice in enumerate(energy_slices)]
    chunksize = chunksize if chunksize is not None else "auto"
    results = Parallel(n_jobs=pool_size, batch_size=chunksize)(delayed(slice_fitter)(x) for x in enumerate(energy_slices))

    slice_params, slice_err = unpack_slice_fit_pool_results(results)

    df = pd.DataFrame(slice_params, columns=FIT_COLUMN_NAMES)
    err_df = pd.DataFrame(slice_err, columns=FIT_ERROR_COLUMN_NAMES)

    return df, err_df

In [ ]:
def two_point_inv_lerp(y: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    deltas = tuple([n2 - n1 for n1, n2 in zip(p1, p2)])
    m = deltas[1] / deltas[0]
    x1, y1 = p1
    if m == 0:
        return x1
    x = (y - y1) / m + x1
    return x


def calculate_q_fom_critical(
    fit_df: pd.DataFrame,
    # q_limits: tuple[float, float] | None = None
    # q_limit_hi: float | None = None
) -> tuple[float | None, float | None]:
    fom_crit = 1.27
    # if q_limits is None:
    #     q_limits = (2500, 60000)  # x axis area with clean FOM curve
    # if q_limit_hi is None:
    #     q_limit_hi = 60000
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    stable_end_idx = np.argmax(fom_x >= 20000)

    # use moving average of delta_y to find stable region (delta_y <= threshold)
    # find first cross in stable region
    window = 5
    # bottleneck window functions use look-behind windows and fill missing with nan
    # so first window-1 values are always nan; we need to convert to look-ahead
    # 
    delta_y_mov_max = bottleneck.move_max(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    delta_y_mov_min = bottleneck.move_min(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    stable_max_delta = delta_y_mov_max < 0.25
    stable_min_delta = delta_y_mov_min > -0.25
    stable_delta = (stable_max_delta & stable_min_delta)
    if not stable_delta.any():
        return None, None
    stable_start_idx = np.argmax(stable_delta)
    stable_start_x = fom_x[stable_start_idx]
    cross_search_slice = fom_y[stable_start_idx:stable_end_idx]
    if not (cross_search_slice >= fom_crit).any():
        return None, stable_start_x
    # argmax gets i in slice, need to add slice start to get i in original list
    fom_critical_idx = np.argmax(cross_search_slice >= fom_crit) + stable_start_idx
    
    # possible_crosses = np.where((fom_y[1:] >= 1.27) & (fom_y[:-1] <= 1.27))[0]
    # # dd_sums = []
    # fom_critical_idx = None
    # for possible_cross in possible_crosses:
    #     pass  # STUB
        # slice_lo = possible_cross - 3 if possible_cross - 3 >= 0 else 0
        # slice_hi = possible_cross + 2
        # deltas = delta_y[slice_lo:slice_hi]
        # delta_deltas = deltas[1:] - deltas[:-1]
        # dd_sum = abs(delta_deltas).sum()
        # dd_sums.append(dd_sum)
    
    # idx_best_cross = np.argmin(np.nan_to_num(dd_sums, nan=np.inf))
    # fom_critical_idx = possible_crosses[idx_best_cross] + 1
    if fom_critical_idx is None:
        q_fom_critical = None
    elif fom_critical_idx > 0:
        # x_crit_bounds = tuple([fom_x[fom_critical_idx+x] for x in [-1, 0]])
        # y_crit_bounds = tuple([fom_y[fom_critical_idx+x] for x in [-1, 0]])
        p1 = fom_x[fom_critical_idx-1], fom_y[fom_critical_idx-1]
        p2 = fom_x[fom_critical_idx], fom_y[fom_critical_idx]
        if p1[1] > fom_crit:  # we can't make lerp extrapolate!
            q_fom_critical = None
        else:
            q_fom_critical = two_point_inv_lerp(fom_crit, p1, p2)
    else:
        q_fom_critical = fom_x[fom_critical_idx]
    return q_fom_critical, stable_start_x

In [ ]:
def calculate_q_fom_critical_from_pulses(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[float | None, float | None]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df)
    fit_df, _ = scan_histogram_slices(histogram, energy_bin_edges, psd_bin_edges)
    q_fom_critical = calculate_q_fom_critical(fit_df)
    return q_fom_critical


def generate_search_grid(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
) -> np.ndarray:
    if isinstance(counts, tuple):
        t1_counts, t2_counts, t3_counts = counts
    else:
        t1_counts, t2_counts, t3_counts = counts, counts, counts
    t1_range = np.linspace(*t1_limits, num=t1_counts)
    t2_range = np.linspace(*t2_limits, num=t2_counts)
    t3_range = np.linspace(*t3_limits, num=t3_counts)
    t_grid = np.meshgrid(t1_range, t2_range, t3_range)
    t_grid = tuple([np.ravel(grid_element) for grid_element in t_grid])
    t_grid_stacked = np.vstack(t_grid)  # shape (3, n)
    return t_grid_stacked


T = TypeVar("T")


def search_grid(
    grid_search_fn: Callable[[np.ndarray, pd.DataFrame], T],
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[np.ndarray, list[T]]:
    t_grid_stacked = generate_search_grid(
        t1_limits, t2_limits, t3_limits, counts
    )

    pool_size = max(
        2 * cores, 4
    )
    # based on https://jupyter-tutorial.readthedocs.io/en/stable
    # /performance/multiprocessing.html
    if use_chunks:
        chunksize, extra = divmod(len(t_grid_stacked.shape[1]), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = "auto"

    parallelizer = Parallel(
        n_jobs=pool_size, batch_size=chunksize, max_nbytes=1e6, verbose=10
    )
    loop_result = parallelizer(
        delayed(grid_search_fn)(timesarray, pulses)
        for timesarray in t_grid_stacked.T
    )
    return t_grid_stacked, loop_result


def grid_search_fom(
    t1_limits: tuple[float, float],
    t2_limits: tuple[float, float],
    t3_limits: tuple[float, float],
    counts: int | tuple[int, int, int],
    pulses: pd.DataFrame,
    cores: int = 4,
    use_chunks: bool = False,
) -> np.ndarray:
    t_grid_stacked, fom_values = search_grid(
        calculate_q_fom_critical_from_pulses,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses,
        cores,
        use_chunks
    )
    
    threshold_energies, threshold_search_starts = list(zip(*fom_values))
    threshold_energies = np.array(threshold_energies, dtype=float)
    threshold_search_starts = np.array(threshold_search_starts, dtype=float)

    return np.vstack(
        [t_grid_stacked, threshold_energies, threshold_search_starts],
        dtype=float
    ).T  # shape (n, 5)

In [ ]:
def calculate_plot_grid_dimensions(n: int, max_cols: int = 4) -> tuple[int, int]:
    ncols = min(n, max_cols)
    nrows = ceil(n / ncols)
    return (nrows, ncols)

In [ ]:
def display_plot_grid(
    grid_plot_fn: Callable[[mpl.axes.Axes, T], None],
    grid_plot_data: list[T],
    grid_count: int,
    max_cols: int
) -> tuple[mpl.figure.Figure, np.ndarray[mpl.axes.Axes]]:
    nrows, ncols = calculate_plot_grid_dimensions(grid_count, max_cols=max_cols)
    fig, axs = plt.subplots(
        nrows, ncols, figsize=(8*ncols, 8*nrows)
    )
    axs = axs.flatten()
    for ax, plot_data in zip(axs, grid_plot_data):
        grid_plot_fn(ax, plot_data)
    return fig, axs

In [ ]:
def pretty_format_duration(duration: float) -> str:
    out_seconds = duration % 60
    dur_minutes = int(duration / 60)
    if dur_minutes == 0:
        return f"{out_seconds:.1f} s"
    out_minutes = dur_minutes % 60
    dur_hours = int(dur_minutes / 60)
    if dur_hours == 0:
        return f"{out_minutes} m {out_seconds:.1f} s"
    else:
        return f"{dur_hours} h {out_minutes} m {out_seconds:.1f} s"

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    # exp_data["signals_df"] = load.load_caen_csvs(exp_id, get_signals=True, get_psd=False)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    signals_df = signals_df.astype("uint32")
    exp_data["signals_df"] = signals_df

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## FOM calculation

In [ ]:
t1 = 95.5
t2 = 111.3
t3 = 240

for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    psd_df = integrate_pulses(signals_df, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    # print(psd_df)
    # print(psd_df["PSD"].max())
    # print(psd_df["PSD"].min())
    exp_data["psd_df"] = psd_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_df = exp_data["psd_df"]
    Z, xe, ye = get_psd_energy_histogram(psd_df, psd_max=1.0)
    histo_data = {"histo": Z, "energy_edges": xe, "psd_edges": ye}
    exp_data["histo_data"] = histo_data

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    histo_data = exp_data["histo_data"]
    fit_df, fit_error_df = scan_histogram_slices(
        histo_data["histo"], histo_data["energy_edges"], histo_data["psd_edges"]
    )
    # print(fit_df.head())
    # print(fit_error_df.head())
    exp_data["fit_df"] = fit_df
    exp_data["fit_error_df"] = fit_error_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    q_fom_critical = calculate_q_fom_critical(fit_df)
    print(q_fom_critical)
    exp_data["q_fom_critical"] = q_fom_critical

### Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
plot_limit = 1000

idx_1 = ceil(t1 / 2)
idx_2 = ceil(t2 / 2)
idx_3 = ceil(t3 / 2)

for exp_id, exp_data in experiment_neutron_data.items():
    signals_df = exp_data["signals_df"]
    fig, ax = plt.subplots(figsize=(24,12))
    row_x = signals_df.columns.map(int)
    for i, (row_id, row_data) in enumerate(signals_df.iterrows()):
        if i > plot_limit:
            break
        # row_x = row_data.index.map(int)
        row_y = row_data.values
        ax.plot(row_x, row_y, "-", alpha=0.1)
        ax.vlines([idx_1, idx_2, idx_3], 15000, 0, linestyles='dashed')
        ax.xaxis.set_major_formatter(lambda x, pos: str(x * 2))

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_df = exp_data["psd_df"]
    fig, ax = plt.subplots(figsize=(14,12))
    ax.plot(psd_df["Q_SHORT"], psd_df["Q_LONG"], ".", alpha=0.01)
    ax.xaxis.set_major_formatter(lambda x, pos: str(x // 1000))
    ax.yaxis.set_major_formatter(lambda x, pos: str(x // 1000))
    ax.set_xlabel("$Q_{short}$ (ADC channels x$10^3$)", fontsize=fontsize)
    ax.set_ylabel("$Q_{long}$ (ADC channels x$10^3$)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_df = exp_data["psd_df"]
    fig, ax = plt.subplots(figsize=(14,12))
    ax.plot(psd_df["Q_LONG"], psd_df["PSD"], ".", alpha=0.1)
    ax.xaxis.set_major_formatter(lambda x, pos: str(x // 1000))
    ax.set_xlabel("$Q_{long}$ (ADC channels x$10^3$)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_ylim(0, 0.8)
    ax.set_xlim(0, 150000)
    ax.tick_params(labelsize=fontsize)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    q_fom_critical, search_start_x = exp_data["q_fom_critical"]
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    delta_x = fom_x[1:-1]
    
    fig, ax = plt.subplots(figsize=(14,12))
    ax_right = ax.twinx()
    ax.plot(fom_x, fom_y, ".-")
    ax_right.plot(delta_x, delta_y, ".:")
    ax.hlines([1.27], 0, 140000, linestyles="dashed")
    ax.vlines([q_fom_critical], 0, 5, linestyles="dashed", alpha=0.5)
    ax.vlines([search_start_x], 0, 5, linestyles="dashdot", alpha=0.5)
    ax.set_xlabel("Slice $Q_{long}$ (ADC channels)", fontsize=fontsize)
    ax.set_ylabel("FOM", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax_right.set_ylabel("Delta FOM", fontsize=fontsize)
    ax_right.tick_params(labelsize=fontsize)
    ax.set_ylim(0, 5)
    # ax.set_xlim(2000, 60000)
    ax.set_xlim(0, 20000)
    ax_right.set_ylim(-5, 5)

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## PSD vs Energy with varying integration times

In [ ]:
t1_range = (70, 100)
t2_range = (102, 130)
t3_range = (200, 380)
t1_setting = 90
t2_setting = 117
t3_setting = 345
counts = 10

for exp_id, exp_data in experiment_neutron_data.items():
    pulses = exp_data["signals_df"]

    psd_df = integrate_pulses(pulses, t1_setting, t2_setting, t3_setting)
    goal_height_variance = abs(psd_df["PEAK_HEIGHT"] - 3500)
    goal_height_row = signals_df.loc[[goal_height_variance.argmin()]]
    # TODO reduce dataframe to just this signal row
    # goal_height_signal = signals_df.loc[goal_height_row.name]
    # print(goal_height_signal)
    
    time_ranges = [t1_range, t2_range, t3_range]
    time_settings = [(t2_setting, t3_setting), (t1_setting, t3_setting), (t1_setting, t2_setting)]
    what_to_vary = ["t1", "t2", "t3"]
    results = {}
    for time_range, time_setting_tuple, varying in zip(time_ranges, time_settings, what_to_vary):
        time_range_values = np.linspace(*time_range, num=counts)
        q_vals: tuple[float, float, float] = []  # input time, Q_long, Q_short
        # print(varying)
        for time_val in time_range_values:
            if varying == "t1":
                t1 = time_val
                t2, t3 = time_setting_tuple
            elif varying == "t2":
                t2 = time_val
                t1, t3 = time_setting_tuple
            elif varying == "t3":
                t3 = time_val
                t1, t2 = time_setting_tuple
            else:
                raise ValueError("bad value for 'varying'")
            psd_df = integrate_pulses(goal_height_row, t1, t2, t3)
            results_row = psd_df.iloc[0]
            q_long = results_row["Q_LONG"]
            q_short = results_row["Q_SHORT"]
            # print(
            #     f"{t1:.1f}/{t2:.1f}/{t3:.1f} - " +
            #     f"{q_long:.1f}/{q_short:.1f}/{(q_long-q_short)/(q_long):.3f}"
            # )
            q_vals.append((time_val, q_long, q_short))
        results[varying] = q_vals
    exp_data["integ_time_test"] = results

### Plotting

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    integ_time_test_results = exp_data["integ_time_test"]
    for time_label, q_vals in integ_time_test_results.items():
        time_vals, q_longs, q_shorts = list(zip(*q_vals))
        psds = [(q_long - q_short) / q_long for q_long, q_short in zip(q_longs, q_shorts)]
        fig, (ax1, ax2, ax3) = plt.subplots(nrows=3, figsize=(12, 32))
        ax1.plot(q_longs, psds, ".-")
        ax2.plot(time_vals, q_longs, ".-")
        ax3.plot(time_vals, psds, ".-")

        for time_val, q_long, psd in zip(time_vals, q_longs, psds):
            ax1.annotate(
                f"{time_val:.1f}",
                xy=(q_long, psd),
                # xytext=(0, 0),
                # textcoords="offset points",
                # ha="center"
            )
        
        ax1.set_title(time_label)
        ax1.set_xlabel("Energy (ADC channels)")
        ax1.set_ylabel("PSD")
        ax2.set_xlabel("Time (ns)")
        ax2.set_ylabel("Energy (ADC channels)")
        ax3.set_xlabel("Time (ns)")
        ax3.set_ylabel("PSD")

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Integration Parameter Optimization

In [ ]:
grid_side_lens = (1, 15, 15)
t1_limits = (95.5, 95.5)
t2_limits = (102, 130)
t3_limits = (200, 380)

In [ ]:
start_time = time.time()
for exp_id, exp_data in experiment_neutron_data.items():
    pulses = exp_data["signals_df"]
    pulses = pulses.astype("uint32")
    fom_search_results = grid_search_fom(
        t1_limits,
        t2_limits,
        t3_limits,
        grid_side_lens,
        pulses
    )
    exp_data["fom_search_results"] = fom_search_results
end_time = time.time()
duration = end_time - start_time
print(pretty_format_duration(duration))

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fom_search_results = exp_data["fom_search_results"]
    print(np.unique(fom_search_results[:, 0]))
    print(np.unique(fom_search_results[:, 1]))
    print(np.unique(fom_search_results[:, 2]))
    *_, row_idx_min_fom, _ = np.nanargmin(fom_search_results, axis=0)
    optimum = fom_search_results[row_idx_min_fom, :]
    print(optimum)
    # print(fom_search_results[:10, :])
    exp_data["optimum"] = optimum

### Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 14

In [ ]:
# TODO set up plotting code
t1_opt = 95.5
t2_opt = 110
t3_opt = 367.14285714
fixed_ts = [t1_opt, t2_opt, t3_opt]
t_idxs = [0, 1, 2]
x_idxs = [1, 0, 0]
y_idxs = [2, 2, 1]
title_ts = ["T1", "T2", "T3"]
title_xs = ["T2", "T1", "T1"]
title_ys = ["T3", "T3", "T2"]

if isinstance(grid_side_lens, tuple):
    t1_side, t2_side, t3_side = grid_side_lens
elif isinstance(grid_side_lens, int):
    t1_side, t2_side, t3_side = [grid_side_lens] * 3
else:
    raise ValueError("Wrong type")

side_lens = {"T1": t1_side, "T2": t2_side, "T3": t3_side}

for exp_id, exp_data in experiment_neutron_data.items():
    fom_search_results = exp_data["fom_search_results"]
    best = exp_data["optimum"]
    fom_best = best[3]
    for params in zip(fixed_ts, t_idxs, x_idxs, y_idxs, title_ts, title_xs, title_ys):
        fixed_t_val, fixed_t_idx, x_idx, y_idx, *rest = params
        title_t, title_x, title_y = rest
        x_best = best[x_idx]
        y_best = best[y_idx]
        mask = fom_search_results[:, fixed_t_idx] == fixed_t_val
        # print(t1_mask)
        fom_masked = fom_search_results[mask]
        # print(fom_t1_opt)

        x_side = side_lens.get(title_x)
        y_side = side_lens.get(title_y)
        if any([
            side is None or side == 1 for side in [x_side, y_side]
        ]):
            continue

        x_vals = fom_masked[:, x_idx].reshape(x_side, x_side)
        y_vals = fom_masked[:, y_idx].reshape(y_side, y_side)
        fom_vals = fom_masked[:, 3].reshape(x_side, y_side)

        fig, ax = plt.subplots(figsize=(12, 16), subplot_kw={"projection": "3d"})
        # TODO rotate plot 180 deg.
        ax.view_init(azim=210)
        ax.plot_surface(x_vals, y_vals, fom_vals, alpha=0.25)
        ax.plot_wireframe(x_vals, y_vals, fom_vals)
        ax.plot([x_best], [y_best], [fom_best], "r*", markersize=16)
        ax.set_title(f"Fixed {title_t} ({fixed_t_val} ns)", fontsize=fontsize)
        ax.set_xlabel(f"{title_x} (ns)", fontsize=fontsize)
        ax.set_ylabel(f"{title_y} (ns)", fontsize=fontsize)
        ax.set_zlabel("Threshold energy (ADC channel x1000)", fontsize=fontsize)
        ax.zaxis.set_major_formatter(lambda z, pos: str(z // 1000))
        ax.tick_params(labelsize=fontsize)
        ax.tick_params(axis="z", pad=2)
        for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
            axis3d.labelpad = 15
        xaxis_ticklabels = ax.xaxis.get_ticklabels()
        for ticklabel in xaxis_ticklabels:
            ticklabel.set_ha("left")
            ticklabel.set_va("center")    
        yaxis_ticklabels = ax.yaxis.get_ticklabels()
        for ticklabel in yaxis_ticklabels:
            ticklabel.set_ha("right")
            ticklabel.set_va("center")
        zaxis_ticklabels = ax.zaxis.get_ticklabels()
        for ticklabel in zaxis_ticklabels:
            ticklabel.set_ha("right")
            ticklabel.set_va("center")
        ax.zaxis.labelpad = 20
        ax.set_box_aspect(None, zoom=0.85)

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## FOM plot grid

In [ ]:
counts = (1, 15, 15)
t1_limits = (95.5, 95.5)
t2_limits = (102, 130)
t3_limits = (200, 380)
# cores = 4
# use_chunks = False

In [ ]:
def calculate_fom_plot_data(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, float | None, float | None]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df)
    fit_df, _ = scan_histogram_slices(histogram, energy_bin_edges, psd_bin_edges)
    q_fom_critical, stable_start_x = calculate_q_fom_critical(fit_df)

    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values

    return (fom_x, fom_y, q_fom_critical, stable_start_x)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    pulses = exp_data["signals_df"]
    t_grid, histogram_data = search_grid(
        calculate_fom_plot_data,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses
    )
    fom_x_list, fom_y_list, threshold_energies, stable_starts = list(zip(*histogram_data))
    
    threshold_energies = np.array(threshold_energies, dtype=float)
    stable_starts = np.array(stable_starts, dtype=float)
    threshold_energy_data = np.vstack([t_grid, threshold_energies, stable_starts], dtype=float).T
    
    exp_data["fom_plot_data"] = (threshold_energy_data, fom_x_list, fom_y_list)

### Plotting

In [ ]:
def plot_fom_in_grid(ax: mpl.axes.Axes, fom_plot_data: tuple[np.ndarray, np.ndarray, np.ndarray]):
    threshold_row, fom_x, fom_y = fom_plot_data
    delta_y = fom_y[2:] - fom_y[:-2]
    delta_x = fom_x[1:-1]
    t1, t2, t3, q_fom_critical, search_start_x = threshold_row

    ax_right = ax.twinx()
    ax.plot(fom_x, fom_y, ".-")
    ax_right.plot(delta_x, delta_y, ".:", alpha=0.5)
    ax.hlines([1.27], 0, 140000, linestyles="dashed")
    ax.vlines([q_fom_critical], 0, 5, linestyles="dashed", alpha=0.5)
    ax.vlines([search_start_x], 0, 5, linestyles="dashdot", alpha=0.5)

    ax.set_title(f"{t1:.1f}/{t2:.1f}/{t3:.1f}")
    ax.set_xlabel("Slice $Q_{long}$ (ADC channels)", fontsize=fontsize)
    ax.set_ylabel("FOM", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax_right.set_ylabel("Delta FOM", fontsize=fontsize)
    ax_right.tick_params(labelsize=fontsize)

    ax.set_ylim(0, 5)
    # ax.set_xlim(2000, 60000)
    ax.set_xlim(0, 20000)
    # ax.set_xlim(10000, 20000)
    ax_right.set_ylim(-0.5, 1)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fom_plot_data = exp_data["fom_plot_data"]
    threshold_energy_data, fom_x_list, fom_y_list = fom_plot_data
    row_count = threshold_energy_data.shape[0]
    if row_count != len(fom_x_list) or row_count != len(fom_y_list):
        print("Unequal rows, cannot plot")
        continue
    grid_plot_data = list(zip(threshold_energy_data, fom_x_list, fom_y_list))

    fig, axs = display_plot_grid(
        plot_fom_in_grid, grid_plot_data, row_count, counts[2]
    )

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Effect of T2 on PSD plot

In [ ]:
counts = (1, 15, 1)
max_cols = 3
t1_limits = (95.5, 95.5)
t2_limits = (102, 130)
t3_limits = (200, 380)

In [ ]:
def calculate_psd_histo_plot_data(
    times: np.ndarray, pulses: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    t1, t2, t3, *_ = times.flatten()
    psd_df = integrate_pulses(pulses, t1, t2, t3)
    psd_df = calculate_psd(psd_df)
    histogram, energy_bin_edges, psd_bin_edges = get_psd_energy_histogram(psd_df, psd_max=1.0)

    energy_bin_mids = (energy_bin_edges[1:] + energy_bin_edges[:-1]) / 2
    psd_bin_mids = (psd_bin_edges[1:] + psd_bin_edges[:-1]) / 2

    return (histogram, energy_bin_mids, psd_bin_mids)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    pulses = exp_data["signals_df"]
    t_grid, histogram_data = search_grid(
        calculate_psd_histo_plot_data,
        t1_limits,
        t2_limits,
        t3_limits,
        counts,
        pulses
    )

    histograms, energy_bin_list, psd_bin_list = list(zip(*histogram_data))
    
    exp_data["histogram_grid_data"] = (t_grid.T, histograms, energy_bin_list, psd_bin_list)

### Plotting

In [ ]:
def plot_histograms_in_grid(
    ax: mpl.axes.Axes, histo_data: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
) -> None:
    t_grid_row, histo_counts, energy_mids, psd_mids = histo_data
    t1, t2, t3, *_ = t_grid_row
    X, Y = np.meshgrid(energy_mids, psd_mids)
    
    ax.pcolormesh(X, Y, histo_counts.T, shading="nearest")
    ax.set_title(f"{t1:.1f}/{t2:.1f}/{t3:.1f}")
    ax.set_xlabel("Pulse Energy (ADC channel)")
    ax.set_ylabel("PSD")

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    t_grid, histograms, energy_bin_list, psd_bin_list = exp_data["histogram_grid_data"]
    row_count = t_grid.shape[0]
    if not all(
        [len(data_list) == row_count
         for data_list in [histograms, energy_bin_list, psd_bin_list]]
    ):
        print("Unequal rows, cannot plot")
        continue
    grid_plot_data = list(zip(t_grid, histograms, energy_bin_list, psd_bin_list))
    fig, axs = display_plot_grid(plot_histograms_in_grid, grid_plot_data, row_count, max_cols)
    # fom_plot_data = exp_data["fom_plot_data"]
    # threshold_energy_data, fom_x_list, fom_y_list = fom_plot_data
    # row_count = threshold_energy_data.shape[0]
    # if row_count != len(fom_x_list) or row_count != len(fom_y_list):
    #     print("Unequal rows, cannot plot")
    #     continue
    # grid_plot_data = list(zip(threshold_energy_data, fom_x_list, fom_y_list))

    # fig, axs = display_plot_grid(
    #     plot_fom_in_grid, grid_plot_data, row_count, counts[2]
    # )

In [ ]:
input("Processing done, hit Enter to finish")
stop()